In [14]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [15]:
# Load CSV
df = pd.read_csv("D:/ISIC_2019_Training_GroundTruth.csv")

# Your classes
classes = [
    "MEL",
    "NV",
    "BCC",
    "AK",
    "BKL",
    "DF",
    "VASC",
    "SCC",
    "UNK"
]

# Convert one-hot label into class number
df["label"] = df[classes].values.argmax(axis=1)

In [16]:
train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["label"],
    random_state=42
)

In [8]:
import os
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset
from torchvision import transforms


class SkinLesionDataset(Dataset):

    def __init__(self, csv_file, image_dir, transform=None):

        self.data = pd.read_csv(csv_file)
        self.image_dir = image_dir
        self.transform = transform

        # Class names
        self.classes = [
            "MEL",
            "NV",
            "BCC",
            "AK",
            "BKL",
            "DF",
            "VASC",
            "SCC",
            "UNK"
        ]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):

        # -------------------------
        # 1. Get image filename
        # -------------------------

        image_name = self.data.iloc[index]["image"]

        if not image_name.endswith(".jpg"):
            image_name += ".jpg"

        image_path = os.path.join(
            self.image_dir,
            image_name
        )

        # -------------------------
        # 2. Load image
        # -------------------------

        image = Image.open(image_path).convert("RGB")

        # -------------------------
        # 3. Get one-hot label
        # -------------------------

        label_vector = self.data.iloc[index][self.classes].values

        # Example:
        # [0, 1, 0, 0, 0, 0, 0, 0, 0]

        # Convert one-hot → class index
        label = label_vector.argmax()

        # Convert to PyTorch tensor
        label = torch.tensor(label, dtype=torch.long)

        # -------------------------
        # 4. Apply transformations
        # -------------------------

        if self.transform:
            image = self.transform(image)

        return image, label

In [17]:
train_df.to_csv("train.csv", index=False)

test_df.to_csv("test.csv", index=False)

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["label"],
    random_state=42
)


In [3]:
print(torch.cuda.is_available())

False


In [1]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)

2.11.0+cu128
True
12.8


In [3]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])


In [20]:
train_dataset = SkinLesionDataset(
    csv_file="D:/diesesedetection/train.csv",
    image_dir="D:/ISIC_2019_Training_Input/ISIC_2019_Training_Input",
    transform=transform
)




test_dataset = SkinLesionDataset(
    csv_file="D:/diesesedetection/test.csv",
    image_dir="D:/ISIC_2019_Training_Input/ISIC_2019_Training_Input",
    transform=transform
)

In [21]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)


test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [27]:
import timm

model = timm.create_model(
    "swin_base_patch4_window7_224",
    pretrained=True,
    num_classes=9,
    drop_path_rate=0.2,
    drop_rate=0.1
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [35]:
model=model.to(device)

In [36]:
for param in model.parameters():
     param.requires_grad = True

In [37]:
criterion = torch.nn.CrossEntropyLoss(label_smoothing=0.1)

In [38]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-5,
    weight_decay=0.01
)

In [39]:
from tqdm.auto import tqdm

def train_one_epoch(model, loader):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc="Training", leave=False)

    for images, labels in pbar:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        correct += (predicted == labels).sum().item()
        total += labels.size(0)

        # Live update
        pbar.set_postfix(
            loss=f"{total_loss/(pbar.n+1):.4f}",
            acc=f"{100*correct/total:.2f}%"
        )

    loss = total_loss / len(loader)
    acc = correct / total

    return loss, acc

In [40]:
def evaluate(model, loader):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            total_loss += loss.item()

            _, predicted = torch.max(outputs,1)

            correct += (predicted==labels).sum().item()

            total += labels.size(0)

    loss = total_loss / len(loader)

    acc = correct / total

    return loss, acc

In [ ]:
epochs = 30
best_model = float("inf")

for epoch in range(epochs):

    print(f"\nEpoch {epoch+1}/{epochs}")

    train_loss, train_acc = train_one_epoch(model, train_loader)

    val_loss, val_acc = evaluate(model, test_loader)

    print(f"Train Loss : {train_loss:.4f}")
    print(f"Train Acc  : {train_acc:.4f}")
    print(f"Val Loss   : {val_loss:.4f}")
    print(f"Val Acc    : {val_acc:.4f}")
    print("-"*40)

    if val_loss < best_model:
        best_model = val_loss
        torch.save(model.state_dict(), "vit_model.pth")
        print("✅ Best model saved!")


Epoch 1/30


Training:  12%|█▏        | 78/634 [00:56<06:36,  1.40it/s, acc=54.29%, loss=1.5402]

In [ ]:
evaluate(model,test_loader)